# Lezione 12 - Riduzione della Cronologia Chat con Agent Scratchpad

Questo notebook mostra come gestire il contesto in conversazioni lunghe usando Microsoft Agent Framework. Con l’aumentare delle conversazioni, il numero di token cresce — superando infine la finestra di contesto del modello. Affrontiamo questo problema con un **pattern di sintesi del contesto** e un **agent scratchpad** per una memoria persistente.

## Cosa Imparerai:
1. **Perché la Gestione del Contesto è Importante**: Comprendere i limiti di token e le finestre di contesto
2. **Agenti Consapevoli del Contesto**: Costruire agenti che gestiscono il proprio contesto di conversazione
3. **Pattern di Sintesi del Contesto**: Usare strumenti per condensare la cronologia delle conversazioni
4. **Agent Scratchpad**: Memoria persistente che sopravvive alla riduzione del contesto

## Prerequisiti:
- Configurazione di Azure OpenAI con variabili di ambiente impostate
- Comprensione dei concetti base di agenti dalle lezioni precedenti


## Configurazione


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Perché la Gestione del Contesto è Importante

Ogni LLM ha una **finestra di contesto** finita — il numero massimo di token che può elaborare in una singola richiesta. Man mano che una conversazione a più turni procede:

- Il **conteggio dei token cresce linearmente** con ogni messaggio dell'utente e risposta dell'assistente.
- I **token del prompt dominano il costo** perché l'intera cronologia viene reinviata ad ogni turno.
- Alla fine la conversazione **supera la finestra di contesto** e il modello tronca o genera un errore.

### Strategie per Gestire il Contesto

| Strategia | Come Funziona | Compromesso |
|---|---|---|
| **Troncamento** | Eliminare i messaggi più vecchi | Perdita del contesto iniziale |
| **Sintesi** | Condensare i messaggi più vecchi in un riassunto | Alcuni dettagli perduti, ma punti chiave conservati |
| **Scratchpad / Memoria Esterna** | Conservare fatti chiave al di fuori della conversazione | Richiede chiamate a strumenti, ma sopravvive a qualsiasi riduzione |

In questo notebook combiniamo la **sintesi** con uno **strumento scratchpad** così che l’agente possa mantenere la continuità anche quando la storia della conversazione è condensata.


## Creazione di un Agente Consapevole del Contesto


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulazione di una lunga conversazione

Facciamo un esempio di conversazione a più turni per vedere come si accumula il contesto. L'agente dovrebbe mantenere i dettagli chiave (preferenze, budget, date di viaggio) attraverso i turni e dimostrare continuità.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Nota come l'agente mantiene il contesto dei turni precedenti — conosce il Giappone, il sushi, i templi, la fotografia, il budget di 3000 $, i viaggi da solo e il periodo di aprile. In una conversazione breve questo funziona bene, ma man mano che la conversazione cresce, inviare nuovamente tutta la storia diventa costoso.

Continuiamo la conversazione con più turni per vedere l'accumulo del contesto:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Modello di Sintesi del Contesto

Man mano che la conversazione cresce, possiamo usare un **strumento di sintesi** per condensare il contesto accumulato in un formato compatto. L'agente chiama questo strumento per registrare le preferenze chiave in modo che, anche se i messaggi più vecchi vengono eliminati, le informazioni essenziali siano preservate.

Questo modello è il blocco di costruzione per una riduzione della storia più sofisticata:
1. L'agente identifica fatti chiave dalla conversazione
2. Chiama lo strumento di sintesi per conservarli
3. I messaggi più vecchi possono essere rimossi in sicurezza perché il riepilogo cattura ciò che conta

Qui sotto definiamo uno strumento `summarize_preferences` che l'agente può chiamare per registrare un riassunto compatto di ciò che ha appreso.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Riepilogo

In questa lezione hai imparato come gestire il contesto nelle conversazioni lungo termine con agenti utilizzando Microsoft Agent Framework:

### Concetti Chiave
- **Le finestre di contesto sono finite** — ogni token nella cronologia della conversazione ha un costo e conta per il limite.
- **Gli strumenti di sintesi** permettono all'agente di condensare il contesto accumulato in riassunti compatti, riducendo l'uso dei token e preservando le informazioni essenziali.
- **I blocchi note degli agenti** forniscono una memoria esterna persistente che sopravvive a qualsiasi riduzione della conversazione.

### Cosa Hai Costruito
- Un **agente consapevole del contesto** che mantiene la continuità nelle conversazioni a più turni
- Uno **strumento di sintesi** (`summarize_preferences`) che registra i dettagli chiave dell'utente in un formato compatto
- Una **conversazione a più turni** che dimostra la conservazione del contesto e la gestione dei cambiamenti

### Applicazioni nel Mondo Reale
- **Bot di assistenza clienti**: Ricordano le preferenze durante lunghe sessioni di supporto
- **Assistenti personali**: Tracciano progetti in corso senza dover ri-spiegare il contesto
- **Tutor educativi**: Mantengono il progresso dello studente attraverso molte interazioni

### Passi Successivi
- Implementare un completo strumento blocco note con persistenza basata su file
- Aggiungere troncamento automatico della cronologia dopo la sintesi
- Combinare con database vettoriali per la ricerca semantica nella memoria
- Costruire agenti che possono riprendere conversazioni anche dopo giorni con il contesto completo


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
